# EstimIA — Analyse exploratoire des données (EDA)
**Département : Paris (75)**

Ce notebook sert à :
1. Vérifier la qualité du dataset produit par `pipeline.py`
2. Visualiser la distribution des prix et des features
3. Préparer les slides de présentation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

# Charger le dataset propre
DATA_PATH = Path('../data/processed/dataset_propre_75.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['date_mutation'])
print(f'Dataset chargé : {len(df):,} transactions')
df.head()

## 1. Vue d'ensemble du dataset

In [ ]:
# Statistiques descriptives principales
print('=== Statistiques générales ===')
print(f"Période : {df['annee'].min():.0f} – {df['annee'].max():.0f}")
print(f"Transactions : {len(df):,}")
print(f"Appartements : {(df['type_bien']=='Appartement').sum():,}")
print(f"Maisons      : {(df['type_bien']=='Maison').sum():,}")
print()

desc = df[['prix', 'surface_m2', 'prix_m2', 'nb_pieces']].describe().round(1)
display(desc)

In [ ]:
# Valeurs manquantes
missing = df.isna().sum()
missing = missing[missing > 0]
if missing.empty:
    print('Aucune valeur manquante — dataset propre.')
else:
    print('Valeurs manquantes :')
    print(missing)

## 2. Distribution des prix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution prix total
axes[0].hist(df['prix'] / 1e6, bins=60, color='#378ADD', edgecolor='white', linewidth=0.4)
axes[0].set_title('Distribution des prix de vente')
axes[0].set_xlabel('Prix (millions €)')
axes[0].set_ylabel('Nombre de transactions')
median_prix = df['prix'].median() / 1e6
axes[0].axvline(median_prix, color='#D85A30', linestyle='--', linewidth=1.5, label=f'Médiane : {median_prix:.2f}M€')
axes[0].legend()

# Distribution prix au m²
axes[1].hist(df['prix_m2'], bins=60, color='#1D9E75', edgecolor='white', linewidth=0.4)
axes[1].set_title('Distribution du prix au m²')
axes[1].set_xlabel('Prix au m² (€)')
axes[1].set_ylabel('Nombre de transactions')
median_m2 = df['prix_m2'].median()
axes[1].axvline(median_m2, color='#D85A30', linestyle='--', linewidth=1.5, label=f'Médiane : {median_m2:,.0f} €/m²')
axes[1].legend()

plt.suptitle('Paris (75) — Distribution des prix', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../docs/fig_distribution_prix.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Prix au m² par arrondissement

In [ ]:
# Extraire l'arrondissement depuis le code postal (752XX)
df['arrondissement'] = df['code_postal'].astype(str).str[-2:].str.lstrip('0')
df['arrondissement'] = df['arrondissement'].replace('', '1')  # 75001 → '1'

prix_arr = (
    df.groupby('arrondissement')['prix_m2']
    .median()
    .reset_index()
    .sort_values('arrondissement', key=lambda x: x.astype(int))
)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(
    prix_arr['arrondissement'],
    prix_arr['prix_m2'],
    color='#378ADD',
    edgecolor='white',
    linewidth=0.4
)

# Colorer le plus cher en orange
max_idx = prix_arr['prix_m2'].idxmax()
arr_vals = prix_arr['arrondissement'].tolist()
for i, (arr, bar) in enumerate(zip(prix_arr['arrondissement'], bars)):
    if prix_arr.loc[prix_arr['arrondissement']==arr, 'prix_m2'].values[0] == prix_arr['prix_m2'].max():
        bar.set_color('#D85A30')

ax.set_title('Prix médian au m² par arrondissement — Paris', fontsize=13, fontweight='bold')
ax.set_xlabel('Arrondissement')
ax.set_ylabel('Prix médian au m² (€)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f} €'))

plt.tight_layout()
plt.savefig('../docs/fig_prix_arrondissement.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Carte de chaleur des transactions (Plotly)

In [ ]:
# Carte interactive des prix au m² géolocalisés
# Échantillon pour la lisibilité (max 5000 points)
sample = df.sample(min(5000, len(df)), random_state=42)

fig = px.scatter_mapbox(
    sample,
    lat='latitude',
    lon='longitude',
    color='prix_m2',
    size='surface_m2',
    color_continuous_scale='RdYlGn_r',
    range_color=[sample['prix_m2'].quantile(0.05), sample['prix_m2'].quantile(0.95)],
    hover_data={
        'prix': ':,.0f',
        'prix_m2': ':,.0f',
        'surface_m2': ':.0f',
        'type_bien': True,
        'nb_pieces': True,
    },
    mapbox_style='carto-positron',
    zoom=11,
    center={'lat': 48.8566, 'lon': 2.3522},
    title='Prix au m² — Paris (75)',
    labels={'prix_m2': 'Prix/m² (€)'},
    height=550,
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.write_html('../docs/carte_prix_paris.html')
fig.show()

## 5. Corrélations features → prix

In [ ]:
# Matrice de corrélation
num_cols = ['prix', 'prix_m2', 'surface_m2', 'nb_pieces',
            'score_dpe_median', 'score_georisques', 'score_delinquance']
num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    mask=mask,
    ax=ax,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8},
)
ax.set_title('Corrélations entre les features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/fig_correlations.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Évolution des prix dans le temps

In [ ]:
prix_temps = (
    df.groupby(['annee', 'type_bien'])['prix_m2']
    .median()
    .reset_index()
)

fig = px.line(
    prix_temps,
    x='annee',
    y='prix_m2',
    color='type_bien',
    markers=True,
    title='Évolution du prix médian au m² — Paris',
    labels={'prix_m2': 'Prix médian au m² (€)', 'annee': 'Année', 'type_bien': 'Type'},
    color_discrete_map={'Appartement': '#378ADD', 'Maison': '#D85A30'},
)
fig.update_layout(yaxis_tickformat=',.0f')
fig.write_html('../docs/fig_evolution_prix.html')
fig.show()

## 7. Résumé — Ce que le dataset permet de prédire

Le dataset `dataset_propre_75.csv` contient :
- Les **features d'entrée** pour le modèle ML : surface, nb pièces, localisation GPS, DPE, géorisques, délinquance
- La **variable cible** : `prix` (et `prix_m2`)

**Prochaine étape** : `02_regression_experiments.ipynb` — entraînement du modèle Random Forest.